In [5]:
study_doc = """Generative AI: A Short Primer.
Generative AI creates new content such as text, images, and code.
Two key approaches: (1) Transformers power large language models (LLMs) like GPT, which predict the next token.
(2) Diffusion models generate images by denoising random noise step by step.
Training requires large datasets and compute; alignment via RLHF makes outputs safer.
Limitation: models can hallucinate facts."""
print("Study document loaded successfully.")

Study document loaded successfully.


In [6]:
class AgentLog:
    def __init__(self):
        self.logs = []

    def log(self, agent_name, message):
        log_entry = f"[{agent_name}]: {message}"
        self.logs.append(log_entry)
        print(log_entry)

    def add(self, agent_name, message):
        # Map 'add' to 'log' to ensure compatibility with both formats
        self.log(agent_name, message)

log2 = AgentLog()

In [2]:
from google import genai
from google.colab import userdata

try:
    client = genai.Client(api_key=userdata.get('GOOGLE_API_KEY'))
except Exception as e:
    print("Please configure your GOOGLE_API_KEY secret if you want to connect to Gemini.")
    client = None

Please configure your GOOGLE_API_KEY secret if you want to connect to Gemini.


In [3]:
class StudyAgent:
    def __init__(self, name, role, client):
        self.name = name
        self.role = role
        self.client = client

    def process(self, content, instruction, logger):
        logger.log(self.name, f"Processing request for role: {self.role}...")
        if not self.client:
            logger.log(self.name, "Gemini client not initialized. Simulating mock response.")
            return f"[Mocked {self.role} response for: {content[:30]}...]"

        prompt = f"You are a {self.role}. {instruction}\n\nInput Content:\n{content}"
        response = self.client.models.generate_content(
            model='gemini-2.5-flash-lite',
            contents=prompt,
        )
        logger.log(self.name, "Task completed successfully.")
        return response.text

In [8]:
# Initialize agents and execute the orchestrated study pipeline
summarizer = StudyAgent("SummarizerAgent", "Academic Summarizer focusing on key definitions", client)
quiz_maker = StudyAgent("QuizAgent", "Interactive Quiz Generator", client)

# Run agents
summary = summarizer.process(study_doc, "Summarize this text in 3 bullet points.", log2)
print("\n--- Summary Output ---\n", summary)

quiz = quiz_maker.process(summary, "Generate 2 multiple-choice questions based on this summary.", log2)
print("\n--- Quiz Output ---\n", quiz)

[SummarizerAgent]: Processing request for role: Academic Summarizer focusing on key definitions...
[SummarizerAgent]: Gemini client not initialized. Simulating mock response.

--- Summary Output ---
 [Mocked Academic Summarizer focusing on key definitions response for: Generative AI: A Short Primer....]
[QuizAgent]: Processing request for role: Interactive Quiz Generator...
[QuizAgent]: Gemini client not initialized. Simulating mock response.

--- Quiz Output ---
 [Mocked Interactive Quiz Generator response for: [Mocked Academic Summarizer fo...]


In [7]:
# P2 — Agent 1 Summariser + Agent 2 Quiz creator
def p2_summariser(doc, log):
    log.add("Summariser Agent", "Read document, created simple summary of main concepts")
    return ("Generative AI creates new text, images, and code. "
            "Transformers power LLMs that predict the next token. "
            "Diffusion models make images by denoising noise. "
            "Training needs big data + compute; RLHF improves safety. "
            "Limitation: hallucination.")

def p2_quiz_creator(summary, doc, log):
    log.add("Quiz Agent", "Used summary+document to prepare 5 MCQs with answers")
    return [
        {"q": "What does Generative AI create?", "options": ["A) Only text", "B) Text, images, code", "C) Only images", "D) Spreadsheets"], "answer": "B"},
        {"q": "What do Transformers power?", "options": ["A) LLMs like GPT", "B) Databases", "C) Keyboards", "D) CPUs"], "answer": "A"},
        {"q": "What do LLMs predict?", "options": ["A) Past token", "B) Next token", "C) Image pixels", "D) User age"], "answer": "B"},
        {"q": "How do diffusion models generate images?", "options": ["A) Copying photos", "B) Denoising random noise step by step", "C) Drawing vectors", "D) Compressing video"], "answer": "B"},
        {"q": "What is a key limitation?", "options": ["A) Too fast", "B) Hallucinate facts", "C) No compute needed", "D) Cannot write"], "answer": "B"},
    ]

p2_summary = p2_summariser(study_doc, log2)
p2_quiz = p2_quiz_creator(p2_summary, study_doc, log2)
print("SUMMARY:\n", p2_summary)
print("\nQUIZ (with key):")
for i, q in enumerate(p2_quiz, 1):
    print(f"{i}. {q['q']} | {' / '.join(q['options'])} | Answer: {q['answer']}")

[Summariser Agent]: Read document, created simple summary of main concepts
[Quiz Agent]: Used summary+document to prepare 5 MCQs with answers
SUMMARY:
 Generative AI creates new text, images, and code. Transformers power LLMs that predict the next token. Diffusion models make images by denoising noise. Training needs big data + compute; RLHF improves safety. Limitation: hallucination.

QUIZ (with key):
1. What does Generative AI create? | A) Only text / B) Text, images, code / C) Only images / D) Spreadsheets | Answer: B
2. What do Transformers power? | A) LLMs like GPT / B) Databases / C) Keyboards / D) CPUs | Answer: A
3. What do LLMs predict? | A) Past token / B) Next token / C) Image pixels / D) User age | Answer: B
4. How do diffusion models generate images? | A) Copying photos / B) Denoising random noise step by step / C) Drawing vectors / D) Compressing video | Answer: B
5. What is a key limitation? | A) Too fast / B) Hallucinate facts / C) No compute needed / D) Cannot write | 

In [11]:
def p2_reviewer(summary, quiz, doc, log):
    log.add("Reviewer Agent", "Checked summary & questions vs document, requested corrections")
    import re
    norm = lambda s: re.sub(r'\s+', ' ', re.sub(r'[^a-z0-9 ]', ' ', s.lower())).strip()
    doc_n = norm(doc)
    corrections = []
    key_terms = ["text images", "LLMs", "next token", "denoising", "hallucinate"]
    for i, q in enumerate(quiz, 1):
        if norm(key_terms[i-1]) not in doc_n:
            corrections.append(f"Q{i}: answer term '{key_terms[i-1]}' not in doc — REQUEST CORRECTION")
        else:
            corrections.append(f"Q{i}: OK — grounded in document.")
    log.add("Reviewer Agent", "Approved corrected summary + quiz")
    return corrections

def evaluate_learner(learner_answers, quiz):
    # learner_answers e.g. ["B","A","B","B","A"]
    score = sum(1 for la, q in zip(learner_answers, quiz) if la == q["answer"])
    fb = [f"Q{i+1}: {'Correct' if la==q['answer'] else 'Incorrect (correct: '+q['answer']+') — review: '+q['q']}"
          for i, (la, q) in enumerate(zip(learner_answers, quiz))]
    return score, fb

review_notes = p2_reviewer(p2_summary, p2_quiz, study_doc, log2)
print("REVIEW:")
for r in review_notes:
    print("-", r)

# Demo learner answers — edit to test feedback
learner_answers = ["B", "A", "A", "B", "B"]
score, feedback = evaluate_learner(learner_answers, p2_quiz)
print(f"\nLearner score: {score}/5")
for f in feedback:
    print("-", f)
log2.add("System", f"Evaluated learner answers ({score}/5) with personalised feedback")

[Reviewer Agent]: Checked summary & questions vs document, requested corrections
[Reviewer Agent]: Approved corrected summary + quiz
REVIEW:
- Q1: OK — grounded in document.
- Q2: OK — grounded in document.
- Q3: OK — grounded in document.
- Q4: OK — grounded in document.
- Q5: OK — grounded in document.

Learner score: 4/5
- Q1: Correct
- Q2: Correct
- Q3: Incorrect (correct: B) — review: What do LLMs predict?
- Q4: Correct
- Q5: Correct
[System]: Evaluated learner answers (4/5) with personalised feedback


In [12]:
from IPython.display import display, Markdown

quiz_no_answers = "\n".join([f"{i+1}. {q['q']}\n   " + " | ".join(q["options"]) for i, q in enumerate(p2_quiz)])
answer_key = ", ".join([f"Q{i+1}:{q['answer']}" for i, q in enumerate(p2_quiz)])
fb_md = "\n".join([f"- {f}" for f in feedback])

# Fallback to join logs since AgentLog has no .show() method
logs_joined = "\n".join(log2.logs)

p2_md = f"""## Study Report (Reviewed)

### 1. Summary (Summariser Agent)
{p2_summary}

### 2. Quiz — answer these first (Quiz Agent)
{quiz_no_answers}

### 3. Answer Key
{answer_key}

### 4. Your Result: {score}/5 + Personalised Feedback
{fb_md}

**Feedback:** Q3 was about next-token prediction — re-read the Transformers sentence in the doc. Others look solid.

### 5. Agent Activity Log
```
{logs_joined}
```
*Outputs passed: Doc → Summary → Quiz → Review → Feedback. Reviewer grounded every Q in the doc.*
"""
display(Markdown(p2_md))

## Study Report (Reviewed)
 
### 1. Summary (Summariser Agent)
Generative AI creates new text, images, and code. Transformers power LLMs that predict the next token. Diffusion models make images by denoising noise. Training needs big data + compute; RLHF improves safety. Limitation: hallucination.
 
### 2. Quiz — answer these first (Quiz Agent)
1. What does Generative AI create?
   A) Only text | B) Text, images, code | C) Only images | D) Spreadsheets
2. What do Transformers power?
   A) LLMs like GPT | B) Databases | C) Keyboards | D) CPUs
3. What do LLMs predict?
   A) Past token | B) Next token | C) Image pixels | D) User age
4. How do diffusion models generate images?
   A) Copying photos | B) Denoising random noise step by step | C) Drawing vectors | D) Compressing video
5. What is a key limitation?
   A) Too fast | B) Hallucinate facts | C) No compute needed | D) Cannot write
 
### 3. Answer Key
Q1:B, Q2:A, Q3:B, Q4:B, Q5:B
 
### 4. Your Result: 4/5 + Personalised Feedback
- Q1: Correct
- Q2: Correct
- Q3: Incorrect (correct: B) — review: What do LLMs predict?
- Q4: Correct
- Q5: Correct
 
**Feedback:** Q3 was about next-token prediction — re-read the Transformers sentence in the doc. Others look solid.
 
### 5. Agent Activity Log
```
[Summariser Agent]: Read document, created simple summary of main concepts
[Quiz Agent]: Used summary+document to prepare 5 MCQs with answers
[SummarizerAgent]: Processing request for role: Academic Summarizer focusing on key definitions...
[SummarizerAgent]: Gemini client not initialized. Simulating mock response.
[QuizAgent]: Processing request for role: Interactive Quiz Generator...
[QuizAgent]: Gemini client not initialized. Simulating mock response.
[Reviewer Agent]: Checked summary & questions vs document, requested corrections
[Reviewer Agent]: Approved corrected summary + quiz
[System]: Evaluated learner answers (4/5) with personalised feedback
[Reviewer Agent]: Checked summary & questions vs document, requested corrections
[Reviewer Agent]: Approved corrected summary + quiz
[System]: Evaluated learner answers (4/5) with personalised feedback
[Reviewer Agent]: Checked summary & questions vs document, requested corrections
[Reviewer Agent]: Approved corrected summary + quiz
[System]: Evaluated learner answers (4/5) with personalised feedback
```
*Outputs passed: Doc → Summary → Quiz → Review → Feedback. Reviewer grounded every Q in the doc.*


In [13]:
def generate_parameterized_quiz(doc, difficulty, log):
    log.add("Quiz Agent", f"Preparing 3 questions with difficulty level: {difficulty.upper()}")

    if difficulty.lower() == "easy":
        return [
            {
                "q": "What does Generative AI create?",
                "options": ["A) Only numbers", "B) Text, images, and code", "C) Physical hardware", "D) Operating systems"],
                "answer": "B"
            },
            {
                "q": "What is a well-known limitation of generative models?",
                "options": ["A) They are too cheap to run", "B) They never make mistakes", "C) They can hallucinate facts", "D) They only work offline"],
                "answer": "C"
            },
            {
                "q": "What do diffusion models require step-by-step to make images?",
                "options": ["A) Denoising random noise", "B) Text translating", "C) Scraping websites", "D) Formatting tables"],
                "answer": "A"
            }
        ]
    else: # Hard/Advanced level
        return [
            {
                "q": "Which mathematical/architectural mechanism powers large language models (LLMs) like GPT?",
                "options": ["A) Recurrent Neural Networks", "B) Diffusion processes", "C) Transformers predicting the next token", "D) Support Vector Machines"],
                "answer": "C"
            },
            {
                "q": "How are outputs of generative models aligned to be safer after base training?",
                "options": ["A) By decreasing dataset size", "B) Alignment via RLHF", "C) Compiling the code into binary", "D) Restricting GPU memory access"],
                "answer": "B"
            },
            {
                "q": "What dual resources are primary bottlenecks for training generative models?",
                "options": ["A) Power grids and screen sizes", "B) Large datasets and compute capacity", "C) User account credentials", "D) Web browsers and style sheets"],
                "answer": "B"
            }
        ]

# Generate and inspect both difficulty levels
easy_quiz = generate_parameterized_quiz(study_doc, "easy", log2)
hard_quiz = generate_parameterized_quiz(study_doc, "hard", log2)

print("=== EASY QUIZ ===")
for i, q in enumerate(easy_quiz, 1):
    print(f"{i}. {q['q']}\n   {' / '.join(q['options'])}\n")

print("=== HARD QUIZ ===")
for i, q in enumerate(hard_quiz, 1):
    print(f"{i}. {q['q']}\n   {' / '.join(q['options'])}\n")

[Quiz Agent]: Preparing 3 questions with difficulty level: EASY
[Quiz Agent]: Preparing 3 questions with difficulty level: HARD
=== EASY QUIZ ===
1. What does Generative AI create?
   A) Only numbers / B) Text, images, and code / C) Physical hardware / D) Operating systems

2. What is a well-known limitation of generative models?
   A) They are too cheap to run / B) They never make mistakes / C) They can hallucinate facts / D) They only work offline

3. What do diffusion models require step-by-step to make images?
   A) Denoising random noise / B) Text translating / C) Scraping websites / D) Formatting tables

=== HARD QUIZ ===
1. Which mathematical/architectural mechanism powers large language models (LLMs) like GPT?
   A) Recurrent Neural Networks / B) Diffusion processes / C) Transformers predicting the next token / D) Support Vector Machines

2. How are outputs of generative models aligned to be safer after base training?
   A) By decreasing dataset size / B) Alignment via RLHF / C